Codigo util

In [1]:
# Configuración servidor base de datos transaccional
# Recuerde usar Estudiante_i como usuario y la contraseña asigana en el excel de conexión a maquina virtual como contraseña
db_user = 'DB_202515_l_alfonsob2'
db_psswd = '202522409'
source_db_connection_string = 'jdbc:mysql://157.253.236.120:8080/RaSaTransaccional_ETL'

dest_db_connection_string = 'jdbc:mysql://157.253.236.120:8080/DB_202515_l_alfonsob2'

# Driver de conexion
path_jar_driver = 'C:\Program Files (x86)\MySQL\Connector J 8.0\mysql-connector-java-8.0.28.jar'

In [2]:
import os 
from pyspark.sql import functions as F, SparkSession, types as T
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql.functions import udf, col, length, isnan, when, count, regexp_replace
from pyspark.sql.window import Window
from datetime import datetime

In [3]:
#Configuración de la sesión
conf=SparkConf() \
    .set('spark.driver.extraClassPath', path_jar_driver)

spark_context = SparkContext(conf=conf)
sql_context = SQLContext(spark_context)
spark = sql_context.sparkSession

C:\Users\estudiante\anaconda3\envs\Tutoriales\lib\site-packages\pyspark\sql\context.py:79: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  FutureWarning


In [4]:
#spark.stop()


In [5]:
def obtener_dataframe_de_bd(db_connection_string, sql, db_user, db_psswd):
    df_bd = spark.read.format('jdbc')\
        .option('url', db_connection_string) \
        .option('dbtable', sql) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .load()
    return df_bd

def guardar_db(db_connection_string, df, tabla, db_user, db_psswd):
    df.select('*').write.format('jdbc') \
      .mode('append') \
      .option('url', db_connection_string) \
      .option('dbtable', tabla) \
      .option('user', db_user) \
      .option('password', db_psswd) \
      .option('driver', 'com.mysql.cj.jdbc.Driver') \
      .save()

## Transformacion Area de Servicio
Se importaron todos los datos de las fuentes principales

In [6]:
# Tablas origen en la BD transaccional
tabla_areas  = "FuenteAreasDeServicio_ETL"
tabla_planes = "FuentePlanesBeneficio_ETL"

# Leemos desde la BD de origen
df_areas = obtener_dataframe_de_bd(
    source_db_connection_string,
    tabla_areas,
    db_user,
    db_psswd
)

df_planes = obtener_dataframe_de_bd(
    source_db_connection_string,
    tabla_planes,
    db_user,
    db_psswd
)

In [7]:

df_areas.show(5)


+------------------+--------------------+-------------+-----------------+----------+------------+-----+--------+-----+
|IdAreaDeServicio_T|NombreAreaDeServicio|IdGeografia_T|          Condado|    Estado|PoblacionAct| Area|Densidad|Fecha|
+------------------+--------------------+-------------+-----------------+----------+------------+-----+--------+-----+
|         100622017|New Jersey - Medi...|        34005|Burlington County|New Jersey|      464269|805.0|   577.0| 2017|
|         100722019|New Jersey  - Med...|        34023| Middlesex County|New Jersey|      860807|311.0|  2768.0| 2019|
|         100922020|New Jersey - Medi...|        34019| Hunterdon County|New Jersey|      129924|430.0|   302.0| 2020|
|         101012018|New Jersey - Medi...|        34031|   Passaic County|New Jersey|      518117|185.0|  2801.0| 2018|
|         101062020|New Jersey - Medi...|        34031|   Passaic County|New Jersey|      518117|185.0|  2801.0| 2020|
+------------------+--------------------+-------

El dataframe inicial, que se muestra anteriormente contiene todas las columnas relevantes provenientes de la fuente, sin ningún tipo de transformación. A continuación se aplicarán las transformaciones descritas en el diseño de ETL, propuestas para garantizar el cumplimiento de las reglas del negocio

T2.

In [8]:
df_areas_sel = (
    df_areas
      .select(
          F.col("IdAreaDeServicio_T").alias("IdAreaServicioNatural"),
          F.col("NombreAreaDeServicio").alias("NombreAreaServicio"),
          F.col("Fecha").alias("FechaOrigen"),          # año solamente
          F.col("IdGeografia_T").alias("IdGeografia"),
          F.col("Estado"),
          F.col("Condado")
      )
)

# Tipos básicos (dejamos FechaOrigen como STRING)
df_areas_sel = (
    df_areas_sel
      .withColumn("IdAreaServicioNatural", F.col("IdAreaServicioNatural").cast(T.StringType()))
      .withColumn("IdGeografia",          F.col("IdGeografia").cast(T.StringType()))
      .withColumn("FechaOrigen",          F.col("FechaOrigen").cast(T.StringType()))  # año como string
)

# Limpieza y normalización de texto: TRIM + colapsar espacios + UPPER
df_areas_limpio = (
    df_areas_sel
      .withColumn(
          "NombreAreaServicio",
          F.upper(F.regexp_replace(F.trim(F.col("NombreAreaServicio")), r"\s+", " "))
      )
      .withColumn(
          "Estado",
          F.upper(F.regexp_replace(F.trim(F.col("Estado")), r"\s+", " "))
      )
      .withColumn(
          "Condado",
          F.upper(F.regexp_replace(F.trim(F.col("Condado")), r"\s+", " "))
      )
)

# Manejo de nulos/vacíos para Estado y Condado
df_areas_limpio = (
    df_areas_limpio
      .withColumn(
          "Estado",
          F.when(F.col("Estado").isNull() | (F.col("Estado") == ""), F.lit("DESCONOCIDO"))
           .otherwise(F.col("Estado"))
      )
      .withColumn(
          "Condado",
          F.when(F.col("Condado").isNull() | (F.col("Condado") == ""), F.lit("DESCONOCIDO"))
           .otherwise(F.col("Condado"))
      )
)

T3.

In [9]:
w_dedupe = Window.partitionBy("IdAreaServicioNatural").orderBy(F.col("FechaOrigen").desc_nulls_last())

df_areas_dedupe = (
    df_areas_limpio
      .withColumn("rn", F.row_number().over(w_dedupe))
      .filter(F.col("rn") == 1)
      .drop("rn")
)


T4.

In [10]:
df_planes_ids = (
    df_planes
      .select(F.col("IdAreaDeServicio_T").alias("IdAreaServicioNatural"))  # ojo mayúsculas
      .distinct()
)

# left_semi -> se queda con filas de la izquierda que tengan match en la derecha
df_areas_validas = df_areas_dedupe.join(
    df_planes_ids,
    on="IdAreaServicioNatural",
    how="left_semi"
)


T5.

In [11]:
df_areas_geo = (
    df_areas_validas
      .withColumn("SKGeografia", F.concat_ws("-", F.col("Estado"), F.col("Condado")))
)

# Campo IdGrupoArea no tiene fuente en tu diccionario, lo dejamos en NULL
df_areas_geo = df_areas_geo.withColumn(
    "IdGrupoArea",
    F.lit(None).cast(T.IntegerType())
)

T6.

In [12]:
# Fecha de proceso (hoy) y fecha fin de vigencia (31/12/5000)
fecha_proceso = datetime.today().date()
fecha_fin_vigencia = datetime(5000, 12, 31).date()

df_dim_nueva = (
    df_areas_geo
      .withColumn("FechaDesde", F.lit(fecha_proceso).cast(T.DateType()))
      .withColumn("FechaHasta", F.lit(fecha_fin_vigencia).cast(T.DateType()))
      .withColumn("EsActual",   F.lit(1).cast(T.IntegerType()))
)

T7.

In [13]:
dim_area = obtener_dataframe_de_bd(
    dest_db_connection_string,
    "Proyecto_202515_G9.DimAreaServicio",
    db_user,
    db_psswd
)


T8.

In [15]:
# DIMENSION: DimAreaServicio 


from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window


# 1. CARGA DESDE MYSQL
dim_area = obtener_dataframe_de_bd(
    dest_db_connection_string,
    "Proyecto_202515_G9.DimAreaServicio",
    db_user,
    db_psswd
)

print("Columnas reales en MySQL:")
print(dim_area.columns)

# 2. SELECCIÓN SOLO DE COLUMNA
dim_area = dim_area.select(
    "SKAreaServicio",
    "IdAreaServicioNatural",
    "NombreAreaServicio",
    "SKGeografia",
    "FechaDesde",
    "FechaHasta",
    "EsActual"
)

print("\nDimAreaServicio después de seleccionar columnas válidas:")
dim_area.printSchema()
dim_area.show(10, truncate=False)


Columnas reales en MySQL:
['SKAreaServicio', 'IdAreaServicioNatural', 'NombreAreaServicio', 'SKGeografia', 'FechaDesde', 'FechaHasta', 'EsActual']

DimAreaServicio después de seleccionar columnas válidas:
root
 |-- SKAreaServicio: long (nullable = true)
 |-- IdAreaServicioNatural: string (nullable = true)
 |-- NombreAreaServicio: string (nullable = true)
 |-- SKGeografia: long (nullable = true)
 |-- FechaDesde: date (nullable = true)
 |-- FechaHasta: date (nullable = true)
 |-- EsActual: integer (nullable = true)

+--------------+---------------------+----------------------------------------------+-----------+----------+----------+--------+
|SKAreaServicio|IdAreaServicioNatural|NombreAreaServicio                            |SKGeografia|FechaDesde|FechaHasta|EsActual|
+--------------+---------------------+----------------------------------------------+-----------+----------+----------+--------+
|25769803776   |100552020            |NEW JERSEY - MEDICAL91661NJ2340003-0520205039 |450     

## dim geografia

In [29]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
from datetime import datetime


# T1. Cargar fuente

tabla_geografia = "FuenteAreasDeServicio_ETL"

df_geo_src = obtener_dataframe_de_bd(
    source_db_connection_string,
    tabla_geografia,
    db_user,
    db_psswd
)


# T2. Selección y tipificación

df_geo_sel = (
    df_geo_src
      .select(
          F.col("IdGeografia_T").alias("IdGeografiaNatural"),
          F.col("Estado"),
          F.col("Condado"),
          F.col("PoblacionAct").alias("PoblacionActual"),
          F.col("Area"),
          F.col("Densidad"),
          F.col("Fecha").alias("FechaOrigen")
      )
)

df_geo_sel = (
    df_geo_sel
      .withColumn("IdGeografiaNatural", F.col("IdGeografiaNatural").cast(T.StringType()))
      .withColumn("PoblacionActual", F.col("PoblacionActual").cast(T.DoubleType()))
      .withColumn("Area", F.col("Area").cast(T.DoubleType()))
      .withColumn("Densidad", F.col("Densidad").cast(T.DoubleType()))
      .withColumn("FechaOrigen", F.col("FechaOrigen").cast(T.StringType()))
)


# Limpieza texto

df_geo_limpio = (
    df_geo_sel
      .withColumn("Estado", F.upper(F.regexp_replace(F.trim(F.col("Estado")), r"\s+", " ")))
      .withColumn("Condado", F.upper(F.regexp_replace(F.trim(F.col("Condado")), r"\s+", " ")))
)

df_geo_limpio = (
    df_geo_limpio
      .withColumn("Estado", F.when(F.col("Estado").isNull() | (F.col("Estado") == ""), F.lit("DESCONOCIDO")).otherwise(F.col("Estado")))
      .withColumn("Condado", F.when(F.col("Condado").isNull() | (F.col("Condado") == ""), F.lit("DESCONOCIDO")).otherwise(F.col("Condado")))
)


# T3. Deduplicación

w_geo_dedupe = Window.partitionBy("IdGeografiaNatural").orderBy(F.col("FechaOrigen").desc_nulls_last())

df_geo_dedupe = (
    df_geo_limpio
      .withColumn("rn", F.row_number().over(w_geo_dedupe))
      .filter(F.col("rn") == 1)
      .drop("rn")
)


# T4. Conversión medidas

FACTOR_MI2_A_KM2 = 2.58999

df_geo_conv = (
    df_geo_dedupe
      .withColumn("AreaKm2", (F.col("Area") * F.lit(FACTOR_MI2_A_KM2)).cast(T.DoubleType()))
)

df_geo_conv = (
    df_geo_conv
      .withColumn(
          "DensidadHabitantes",
          F.when(
              (F.col("AreaKm2").isNotNull()) &
              (F.col("AreaKm2") > 0) &
              (F.col("PoblacionActual").isNotNull()),
              (F.col("PoblacionActual") / F.col("AreaKm2"))
          ).otherwise(F.lit(None).cast(T.DoubleType()))
      )
)


# T5-T6. Crear SK numérico + vigencias
w_geo = Window.orderBy("IdGeografiaNatural")

fecha_proceso = datetime.today().date()
fecha_fin_vigencia = datetime(5000, 12, 31).date()

df_geo_nueva = df_geo_conv

df_dim_geo_base = (
    df_geo_nueva
      .withColumn("SKGeografia", F.row_number().over(w_geo))
      .withColumn("FechaDesde", F.lit(fecha_proceso).cast(T.DateType()))
      .withColumn("FechaHasta", F.lit(fecha_fin_vigencia).cast(T.DateType()))
      .withColumn("EsActual", F.lit(1).cast(T.IntegerType()))
      .select(
          "SKGeografia",
          "IdGeografiaNatural",
          "Estado",
          "Condado",
          "PoblacionActual",
          "AreaKm2",
          "DensidadHabitantes",
          "FechaDesde",
          "FechaHasta",
          "EsActual"
      )
)


# T9. Insertar fila DESCONOCIDO

schema_dim_geo = df_dim_geo_base.schema

fila_desconocido = [(
    0,                        
    "0",                      
    "DESCONOCIDO",
    "DESCONOCIDO",
    None,
    None,
    None,
    fecha_proceso,
    fecha_fin_vigencia,
    1
)]

df_desconocido = spark.createDataFrame(fila_desconocido, schema=schema_dim_geo)

df_dim_geografia = (
    df_dim_geo_base
      .unionByName(df_desconocido)
      .dropDuplicates(["SKGeografia"])
)

df_dim_geografia.show(20, truncate=False)


# Cargar en MySQL


guardar_db(
    dest_db_connection_string,
    df_dim_geografia,
    'Proyecto_202515_G9.DimGeografia',
    db_user,
    db_psswd
)



+-----------+------------------+-----------+-----------------+---------------+------------------+------------------+----------+----------+--------+
|SKGeografia|IdGeografiaNatural|Estado     |Condado          |PoblacionActual|AreaKm2           |DensidadHabitantes|FechaDesde|FechaHasta|EsActual|
+-----------+------------------+-----------+-----------------+---------------+------------------+------------------+----------+----------+--------+
|0          |0                 |DESCONOCIDO|DESCONOCIDO      |null           |null              |null              |2025-11-20|5000-12-31|1       |
|1          |10001             |DELAWARE   |KENT COUNTY      |184149.0       |2071.9919999999997|88.87534314804306 |2025-11-20|5000-12-31|1       |
|2          |10003             |DELAWARE   |NEW CASTLE COUNTY|571708.0       |1279.45506        |446.8371089172917 |2025-11-20|5000-12-31|1       |
|3          |10005             |DELAWARE   |SUSSEX COUNTY    |247527.0       |3097.6280399999996|79.908561261603

La visualización anterior se muestra la tabla después de todas las transformaciones realizadas con el objetivo de cumplir los requerimientos del negocio. Posteriormente se procede a realizar el cargue de la misma en la base de datos de MySQL

## dim Fecha

T1.

In [37]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from datetime import datetime


# T1. Función para normalizar fechas

def normalizar_fecha(df, col_raw="FechaRaw", col_fecha="Fecha"):
    col_str = F.col(col_raw).cast("string")
    return (
        df.withColumn(
            col_fecha,
            F.when(F.length(col_str) == 4,
                   F.to_date(col_str, "yyyy"))   # año → 01-01
             .otherwise(F.to_date(col_str))     # formato estándar
        )
        .select(col_fecha)
    )


# T2. Cargar fuentes y extraer columna Fecha

tabla_areas   = "FuenteAreasDeServicio_ETL"
tabla_planes  = "FuentePlanesBeneficio_ETL"
tabla_tipos   = "FuenteTiposBeneficio_ETL"

df_areas  = obtener_dataframe_de_bd(source_db_connection_string, tabla_areas, db_user, db_psswd)
df_planes = obtener_dataframe_de_bd(source_db_connection_string, tabla_planes, db_user, db_psswd)
df_tipos  = obtener_dataframe_de_bd(source_db_connection_string, tabla_tipos, db_user, db_psswd)

df_fechas_areas  = df_areas.select(F.col("Fecha").cast("string").alias("FechaRaw"))
df_fechas_planes = df_planes.select(F.col("Fecha").cast("string").alias("FechaRaw"))
df_fechas_tipos  = df_tipos.select(F.col("Fecha").cast("string").alias("FechaRaw"))


# T3. Normalización y unión de todas las fechas

df_fechas_areas_norm  = normalizar_fecha(df_fechas_areas)
df_fechas_planes_norm = normalizar_fecha(df_fechas_planes)
df_fechas_tipos_norm  = normalizar_fecha(df_fechas_tipos)

df_fechas_union = (
    df_fechas_areas_norm
      .union(df_fechas_planes_norm)
      .union(df_fechas_tipos_norm)
)

df_fechas_depuradas = (
    df_fechas_union
      .filter(F.col("Fecha").isNotNull())
      .filter(F.col("Fecha") >= F.lit("1900-01-01").cast("date"))
      .filter(F.col("Fecha") <= F.lit("5000-12-31").cast("date"))
      .distinct()
)


# T4. Generar calendario completo corregido

min_max = df_fechas_depuradas.agg(
    F.min("Fecha").alias("min_fecha"),
    F.max("Fecha").alias("max_fecha")
).collect()[0]

fecha_min = min_max["min_fecha"]
fecha_max = min_max["max_fecha"]

if fecha_min is None or fecha_max is None:
    raise ValueError("No se pudieron determinar fechas mín/max para construir DimFecha.")

df_calendario = (
    spark.createDataFrame([(fecha_min, fecha_max)], ["start", "end"])
         .withColumn("start", F.col("start").cast("date"))
         .withColumn("end",   F.col("end").cast("date"))
         .select(F.explode(F.sequence("start", "end", F.expr("interval 1 day"))).alias("Fecha"))
)

# T5, T6, T7. Construir DimFecha

df_dim_fecha = (
    df_calendario
      .withColumn("SKFecha", F.date_format(F.col("Fecha"), "yyyyMMdd").cast(T.IntegerType()))
      .withColumn("Anio",      F.year("Fecha").cast(T.IntegerType()))
      .withColumn("Mes",       F.month("Fecha").cast(T.IntegerType()))
      .withColumn("Dia",       F.dayofmonth("Fecha").cast(T.IntegerType()))
      .withColumn("Trimestre",
                  ((F.month("Fecha") - F.lit(1)) / F.lit(3) + F.lit(1)).cast(T.IntegerType()))
      .withColumn("Semestre",
                  F.when(F.month("Fecha") <= 6, F.lit(1))
                   .otherwise(F.lit(2)).cast(T.IntegerType()))
      .withColumn("EsInicioDeAnio",
                  F.when((F.month("Fecha") == 1) & (F.dayofmonth("Fecha") == 1),
                         F.lit(1))
                   .otherwise(F.lit(0)).cast(T.IntegerType()))
      .select(
          "SKFecha",
          "Fecha",
          "Anio",
          "Mes",
          "Dia",
          "Trimestre",
          "Semestre",
          "EsInicioDeAnio"
      )
      .dropDuplicates(["SKFecha"])
      .orderBy("Fecha")
)

df_dim_fecha.show(20, truncate=False)


# Cargar a la BD

guardar_db(
    dest_db_connection_string,
    df_dim_fecha,
    "Proyecto_202515_G9.DimFecha",
    db_user,
    db_psswd
)


+--------+----------+----+---+---+---------+--------+--------------+
|SKFecha |Fecha     |Anio|Mes|Dia|Trimestre|Semestre|EsInicioDeAnio|
+--------+----------+----+---+---+---------+--------+--------------+
|20170101|2017-01-01|2017|1  |1  |1        |1       |1             |
|20170102|2017-01-02|2017|1  |2  |1        |1       |0             |
|20170103|2017-01-03|2017|1  |3  |1        |1       |0             |
|20170104|2017-01-04|2017|1  |4  |1        |1       |0             |
|20170105|2017-01-05|2017|1  |5  |1        |1       |0             |
|20170106|2017-01-06|2017|1  |6  |1        |1       |0             |
|20170107|2017-01-07|2017|1  |7  |1        |1       |0             |
|20170108|2017-01-08|2017|1  |8  |1        |1       |0             |
|20170109|2017-01-09|2017|1  |9  |1        |1       |0             |
|20170110|2017-01-10|2017|1  |10 |1        |1       |0             |
|20170111|2017-01-11|2017|1  |11 |1        |1       |0             |
|20170112|2017-01-12|2017|1  |12 |

AnalysisException: Column "Trimestre" not found in schema Some(StructType(StructField(Fecha,DateType,false), StructField(SKFecha,IntegerType,true), StructField(Anio,IntegerType,false), StructField(Mes,IntegerType,false), StructField(Dia,IntegerType,false)))

T2.

In [38]:
tabla_areas   = "FuenteAreasDeServicio_ETL"
tabla_planes  = "FuentePlanesBeneficio_ETL"  
tabla_tipos   = "FuenteTiposBeneficio_ETL"

df_areas = obtener_dataframe_de_bd(
    source_db_connection_string, tabla_areas, db_user, db_psswd
)
df_planes = obtener_dataframe_de_bd(
    source_db_connection_string, tabla_planes, db_user, db_psswd
)
df_tipos = obtener_dataframe_de_bd(
    source_db_connection_string, tabla_tipos, db_user, db_psswd
)

# Nos quedamos solo con la columna Fecha de cada fuente, como string genérico
df_fechas_areas  = df_areas.select(F.col("Fecha").cast("string").alias("FechaRaw"))
df_fechas_planes = df_planes.select(F.col("Fecha").cast("string").alias("FechaRaw"))
df_fechas_tipos  = df_tipos.select(F.col("Fecha").cast("string").alias("FechaRaw"))


T! T2 & T3.

In [39]:

df_fechas_areas_norm  = normalizar_fecha(df_fechas_areas)
df_fechas_planes_norm = normalizar_fecha(df_fechas_planes)
df_fechas_tipos_norm  = normalizar_fecha(df_fechas_tipos)

df_fechas_union = (
    df_fechas_areas_norm
      .union(df_fechas_planes_norm)
      .union(df_fechas_tipos_norm)
)

df_fechas_depuradas = (
    df_fechas_union
      .filter(F.col("Fecha").isNotNull())
      .filter(F.col("Fecha") >= F.lit("1900-01-01").cast("date"))
      .filter(F.col("Fecha") <= F.lit("5000-12-31").cast("date"))
      .distinct()
)

T4.

In [40]:
min_max = df_fechas_depuradas.agg(
    F.min("Fecha").alias("min_fecha"),
    F.max("Fecha").alias("max_fecha")
).collect()[0]

fecha_min = min_max["min_fecha"]
fecha_max = min_max["max_fecha"]

if fecha_min is None or fecha_max is None:
    raise ValueError("No se pudieron determinar fechas mín/max para construir DimFecha.")

df_calendario = (
    spark.createDataFrame([(fecha_min, fecha_max)], ["start", "end"])
         .select(F.explode(F.sequence("start", "end", F.expr("interval 1 day"))).alias("Fecha"))
)

T5, T6 & T7

In [41]:
df_dim_fecha = (
    df_calendario
      .withColumn("SKFecha",
                  F.date_format(F.col("Fecha"), "yyyyMMdd").cast(T.IntegerType()))
      .withColumn("Anio",      F.year("Fecha").cast(T.IntegerType()))
      .withColumn("Mes",       F.month("Fecha").cast(T.IntegerType()))
      .withColumn("Dia",       F.dayofmonth("Fecha").cast(T.IntegerType()))
      .withColumn("Trimestre",
                  ((F.month("Fecha") - F.lit(1)) / F.lit(3) + F.lit(1)).cast(T.IntegerType()))
      .withColumn("Semestre",
                  F.when(F.month("Fecha") <= 6, F.lit(1))
                   .otherwise(F.lit(2)).cast(T.IntegerType()))
      .withColumn("EsInicioDeAnio",
                  F.when((F.month("Fecha") == 1) & (F.dayofmonth("Fecha") == 1),
                         F.lit(1))
                   .otherwise(F.lit(0)).cast(T.IntegerType()))
      .select(
          "SKFecha",
          "Fecha",
          "Anio",
          "Mes",
          "Dia",
          "Trimestre",
          "Semestre",
          "EsInicioDeAnio"
      )
      .dropDuplicates(["SKFecha"])
      .orderBy("Fecha")
)

In [35]:
df_dim_fecha.show(20, truncate=False)


+--------+----------+----+---+---+---------+--------+--------------+
|SKFecha |Fecha     |Anio|Mes|Dia|Trimestre|Semestre|EsInicioDeAnio|
+--------+----------+----+---+---+---------+--------+--------------+
|20170101|2017-01-01|2017|1  |1  |1        |1       |1             |
|20170102|2017-01-02|2017|1  |2  |1        |1       |0             |
|20170103|2017-01-03|2017|1  |3  |1        |1       |0             |
|20170104|2017-01-04|2017|1  |4  |1        |1       |0             |
|20170105|2017-01-05|2017|1  |5  |1        |1       |0             |
|20170106|2017-01-06|2017|1  |6  |1        |1       |0             |
|20170107|2017-01-07|2017|1  |7  |1        |1       |0             |
|20170108|2017-01-08|2017|1  |8  |1        |1       |0             |
|20170109|2017-01-09|2017|1  |9  |1        |1       |0             |
|20170110|2017-01-10|2017|1  |10 |1        |1       |0             |
|20170111|2017-01-11|2017|1  |11 |1        |1       |0             |
|20170112|2017-01-12|2017|1  |12 |

La anterior tabla muestra los datos tal como se cargaron a SQL, son los datos de fecha transformados con la adición de 3 columnas Trimestre, Semestre, EsInicioDeAnio, estas columnas se propusieron como una alternativa que en un futuro podría fácilitar el análisis por periodos de tiempo

In [42]:
guardar_db(
    dest_db_connection_string,
    df_dim_fecha,
    'Proyecto_202515_G9.DimFecha',
    db_user,
    db_psswd
)

AnalysisException: Column "Trimestre" not found in schema Some(StructType(StructField(Fecha,DateType,false), StructField(SKFecha,IntegerType,true), StructField(Anio,IntegerType,false), StructField(Mes,IntegerType,false), StructField(Dia,IntegerType,false)))

## Dim condicion pago

Extraccion

In [43]:

tabla_cond   = "FuenteCondicionesDePago_ETL"
tabla_planes = "FuentePlanesBeneficio_ETL"

df_cond = obtener_dataframe_de_bd(
    source_db_connection_string, tabla_cond, db_user, db_psswd
)

df_planes = obtener_dataframe_de_bd(
    source_db_connection_string, tabla_planes, db_user, db_psswd
)

T1, T2 & T3.

In [44]:
df_cond_sel = (
    df_cond
      .select(
          F.col("IdCondicionesDePago_T").alias("IdCondicionPagoNatural"),
          F.col("Descripcion"),
          F.col("Tipo")
      )
)

df_cond_sel = (
    df_cond_sel
      .withColumn("IdCondicionPagoNatural", F.col("IdCondicionPagoNatural").cast(T.StringType()))
      .withColumn("Descripcion",            F.col("Descripcion").cast(T.StringType()))
      .withColumn("Tipo",                   F.col("Tipo").cast(T.StringType()))
)

df_cond_limpio = (
    df_cond_sel
      .withColumn(
          "Descripcion",
          F.upper(F.regexp_replace(F.trim(F.col("Descripcion")), r"\s+", " "))
      )
      .withColumn(
          "Tipo",
          F.upper(F.regexp_replace(F.trim(F.col("Tipo")), r"\s+", " "))
      )
)

df_cond_limpio = (
    df_cond_limpio
      .filter(F.col("IdCondicionPagoNatural").isNotNull())
      .filter(F.col("IdCondicionPagoNatural") != "")
)

T4.

In [45]:

w_cond_dedupe = (
    Window
      .partitionBy("IdCondicionPagoNatural")
      .orderBy(F.col("Descripcion").asc_nulls_last(), F.col("Tipo").asc_nulls_last())
)

df_cond_dedupe = (
    df_cond_limpio
      .withColumn("rn", F.row_number().over(w_cond_dedupe))
      .filter(F.col("rn") == 1)
      .drop("rn")
)

T5.

In [46]:
# Usando IdCondicionDePagoCopago_T y IdCondicionDePagoCoseguro_T
df_ids_copago = (
    df_planes
      .select(F.col("IdCondicionDePagoCopago_T").alias("IdCondicionPagoNatural"))
      .filter(F.col("IdCondicionPagoNatural").isNotNull())
)

df_ids_coseguro = (
    df_planes
      .select(F.col("IdCondicionDePagoCoseguro_T").alias("IdCondicionPagoNatural"))
      .filter(F.col("IdCondicionPagoNatural").isNotNull())
)

df_ids_usadas = (
    df_ids_copago
      .union(df_ids_coseguro)
      .distinct()
)

df_cond_validas = df_cond_dedupe.join(
    df_ids_usadas,
    on="IdCondicionPagoNatural",
    how="left_semi"
)

T6.

In [47]:
w_sk = Window.orderBy("IdCondicionPagoNatural")

df_dim_condicion_pago = (
    df_cond_validas
      .withColumn("SKCondicionPago", F.row_number().over(w_sk))
      .select(
          "SKCondicionPago",
          "IdCondicionPagoNatural",
          "Descripcion",
          "Tipo"
      )
)


In [48]:
df_dim_condicion_pago.show(20, truncate=False)


+---------------+----------------------+--------------------------------+-----------+
|SKCondicionPago|IdCondicionPagoNatural|Descripcion                     |Tipo       |
+---------------+----------------------+--------------------------------+-----------+
|1              |102                   |COPAY PER STAY BEFORE DEDUCTIBLE|COPAGO     |
|2              |119                   |COPAY PER DAY WITH DEDUCTIBLE   |COPAGO     |
|3              |136                   |COPAY PER STAY WITH DEDUCTIBLE  |COPAGADO   |
|4              |153                   |COPAY AFTER DEDUCTIBLE          |COPAGO     |
|5              |17                    |COPAY PER DAY AFTER DEDUCTIBLE  |COPAGADO   |
|6              |170                   |COPAY BEFORE DEDUCTIBLE         |COPAGO     |
|7              |18                    |NO CHARGE AFTER DEDUCTIBLE      |COSEGURO   |
|8              |187                   |COPAY WITH DEDUCTIBLE           |COPAGO     |
|9              |204                   |COPAY PER DAY 

In [49]:
guardar_db(
    dest_db_connection_string,
    df_dim_condicion_pago,
    'Proyecto_202515_G9.DimCondicionPago',
    db_user,
    db_psswd
)

# Dimensión tipo beneficio

### Extracción

In [50]:
sql_tipo_beneficio = '''
(
SELECT 
    idTipoBeneficio_T AS ID_TipoBeneficio_T,
    Nombre AS NombreBeneficio,
    UnidadDelLimite AS UnidadLimite,
    EsEHB,
    EstaCubiertaPorSeguro,
    TieneLimiteCuantitativo,
    ExcluidoDelDesembolsoMaximoDentroDeLaRed,
    ExcluidoDelDesembolsoMaximoFueraDeLaRed,
    Fecha AS FechaDesde_Fuente
FROM FuenteTiposBeneficio_ETL
) AS Temp_tipo_beneficio
'''

tipo_beneficio = obtener_dataframe_de_bd(
    source_db_connection_string, 
    sql_tipo_beneficio, 
    db_user, 
    db_psswd
)

tipo_beneficio.show(5, truncate=False)

+------------------+-----------------------------------------------+-------------------+-----+---------------------+-----------------------+----------------------------------------+---------------------------------------+-----------------+
|ID_TipoBeneficio_T|NombreBeneficio                                |UnidadLimite       |EsEHB|EstaCubiertaPorSeguro|TieneLimiteCuantitativo|ExcluidoDelDesembolsoMaximoDentroDeLaRed|ExcluidoDelDesembolsoMaximoFueraDeLaRed|FechaDesde_Fuente|
+------------------+-----------------------------------------------+-------------------+-----+---------------------+-----------------------+----------------------------------------+---------------------------------------+-----------------+
|5                 |Abortion For Which Public Funding Is Prohibited|                   |No   |No                   |No                     |No                                      |No                                     |2017             |
|5                 |Abortion For Which P

#Transformaciones 

In [51]:
#T1 Renombrar "ID_TipoBeneficio_T" como "IdTipoBeneficioNatural"
tipo_beneficio = tipo_beneficio.withColumnRenamed("ID_TipoBeneficio_T", "IdTipoBeneficioNatural")

# t2 Crear SK
tipo_beneficio = tipo_beneficio.coalesce(1).withColumn("SKTipoBeneficio", F.monotonically_increasing_id() + 1)

# T3 Validar unicidad de UnidadDelLimite por IdTipoBeneficioNatural y año
tipo_beneficio = tipo_beneficio.withColumn("Anio", F.col("FechaDesde_Fuente"))

validacion_unicidad = tipo_beneficio.groupBy("IdTipoBeneficioNatural", "Anio").agg(
    F.countDistinct("UnidadLimite").alias("UnidadesDistintas")).filter("UnidadesDistintas > 1")

if validacion_unicidad.count() == 0:
    print("Se comprueba que existe unicidad en la UnidadDelLimite para el mismo IdTipoBeneficioNatural en el mismo año, se puede continuar en proceso con normalidad")



Se comprueba que existe unicidad en la UnidadDelLimite para el mismo IdTipoBeneficioNatural en el mismo año, se puede continuar en proceso con normalidad


In [52]:
tipo_beneficio.select("TieneLimiteCuantitativo") \
              .distinct() \
              .orderBy("TieneLimiteCuantitativo") \
              .show(100, truncate=False)

+-----------------------+
|TieneLimiteCuantitativo|
+-----------------------+
|No                     |
|Si                     |
|Yes                    |
+-----------------------+



In [53]:
# T4. Estandarizar los valores del campo "TieneLimiteCuantitativo" para que queden unicamente "Yes" y "No"
tipo_beneficio = tipo_beneficio.withColumn("TieneLimiteCuantitativo",
    F.when(F.col("TieneLimiteCuantitativo") == "Si", "Yes").when(F.col("TieneLimiteCuantitativo") == "Yes", "Yes")
                                           .otherwise("No"))
tipo_beneficio.select("TieneLimiteCuantitativo") \
              .distinct() \
              .orderBy("TieneLimiteCuantitativo") \
              .show(100, truncate=False)


+-----------------------+
|TieneLimiteCuantitativo|
+-----------------------+
|No                     |
|Yes                    |
+-----------------------+



In [54]:
# T5. Crear columna FechaDesde con la fecha de incio de vigencia del registro
tipo_beneficio = tipo_beneficio.withColumn("FechaDesde",F.to_date(F.concat_ws("-", F.col("Anio"), F.lit("01"), F.lit("01")),
        "yyyy-MM-dd"))
tipo_beneficio.select("FechaDesde").show(5, False)


+----------+
|FechaDesde|
+----------+
|2017-01-01|
|2020-01-01|
|2017-01-01|
|2019-01-01|
|2019-01-01|
+----------+
only showing top 5 rows



In [55]:
# T6. Crear columna FechaHasta con la fecha en que algún atributo del beneficio cambia (si el registro está vigente NULL)
tipo_beneficio = tipo_beneficio.withColumn("FechaHasta", F.lit(None).cast("date"))
# T7. Crear Columna EsActual, 1 si el registro está vigente y 0 si es un dato histórico
tipo_beneficio = tipo_beneficio.withColumn("EsActual",F.when(
    (F.col("FechaHasta").isNull()) | (F.col("FechaHasta") >= F.current_date()),1).otherwise(0))
tipo_beneficio.select("FechaDesde", "FechaHasta", "EsActual").show(20, False)

   

+----------+----------+--------+
|FechaDesde|FechaHasta|EsActual|
+----------+----------+--------+
|2017-01-01|null      |1       |
|2020-01-01|null      |1       |
|2017-01-01|null      |1       |
|2019-01-01|null      |1       |
|2019-01-01|null      |1       |
|2018-01-01|null      |1       |
|2020-01-01|null      |1       |
|2017-01-01|null      |1       |
|2018-01-01|null      |1       |
|2017-01-01|null      |1       |
|2017-01-01|null      |1       |
|2020-01-01|null      |1       |
|2020-01-01|null      |1       |
|2017-01-01|null      |1       |
|2020-01-01|null      |1       |
|2017-01-01|null      |1       |
|2018-01-01|null      |1       |
|2019-01-01|null      |1       |
|2021-01-01|null      |1       |
|2017-01-01|null      |1       |
+----------+----------+--------+
only showing top 20 rows



La anterior tabla muestra los datos luego de la transformación del manejo de los datos históricos, la gran cantidad de nulos es porque los primeros 20 registros aún no han sido modificados, es decir que no existe una fecha asociada a su caducidad

In [56]:
# 1. Total de filas
total_filas = tipo_beneficio.count()
print("Total de filas:", total_filas)

# 2. IDs únicos
ids_unicos = tipo_beneficio.select("IdTipoBeneficioNatural").distinct().count()
print("IDs únicos:", ids_unicos)

# 3. IDs repetidos
ids_repetidos = total_filas - ids_unicos
print("IDs repetidos:", ids_repetidos)

Total de filas: 449
IDs únicos: 205
IDs repetidos: 244


Hay bastantes IDs repetidos, si la información de cada fila es la información de un tipo de beneficio no debería existir Ids repetidos,se asume que estos IDs repetidos corresponden a nuevas publicaciones de la información del beneficio por lo que se asume que el registro vigente es el que tiene la fecha más reciente 

In [57]:
#T8. Si cambia algún atributo del plan cerrar el registro anterior (actualizar FechaHasta y EsActual = 0) y crear uno nuevo
w = Window.partitionBy("IdTipoBeneficioNatural").orderBy("FechaDesde")
tipo_beneficio = tipo_beneficio.withColumn("FechaHasta",F.lead("FechaDesde").over(w))
tipo_beneficio = tipo_beneficio.withColumn("FechaHasta",
                F.when(F.col("FechaHasta").isNotNull(), F.expr("date_sub(FechaHasta, 1)")).otherwise(F.lit(None).cast("date")))
tipo_beneficio = tipo_beneficio.withColumn("EsActual",F.when(F.col("FechaHasta").isNull(), 1).otherwise(0))
tipo_beneficio.select("EsActual").distinct().show()


+--------+
|EsActual|
+--------+
|       0|
|       1|
+--------+



In [58]:
#Seleccionar solo las variables que se deben subir a la base de datos
tipo_beneficio_final = tipo_beneficio.select(
    "IdTipoBeneficioNatural",
    "NombreBeneficio",
    "UnidadLimite",
    "EsEHB",
    "EstaCubiertaPorSeguro",
    "TieneLimiteCuantitativo",
    "ExcluidoDelDesembolsoMaximoDentroDeLaRed",
    "ExcluidoDelDesembolsoMaximoFueraDeLaRed",
    "FechaDesde",
    "FechaHasta",
    "EsActual",
    "SKTipoBeneficio"
)

In [59]:
guardar_db(
    dest_db_connection_string,
    tipo_beneficio_final,
    'Proyecto_202515_G9.DimTipoBeneficio',
    db_user,
    db_psswd
)

# Dimensión plan 
## Extración 
En esta dimensión se propuso usar campos que no están en la base de datos pero consideramos que deben existir y que son indispensables para los análisis solicitados por la compañía. En el ejercicio con la empresa se consultaría a la misma y se completaría la base de datos, sin embargo, al ser este un ejercicio académico y no tener contacto con la compañía para solicitar la información se decidió crear sinteticamente con el objetivo de desarrollar el proceso de ETL completo

In [60]:
sql_plan = '''
(
    SELECT idPlan_T
    FROM FuentePlanesBeneficio_ETL
) AS Temp_plan
'''

planes = obtener_dataframe_de_bd(
    source_db_connection_string,
    sql_plan,
    db_user,
    db_psswd
)

planes.show(5)


+-----------------+
|         idPlan_T|
+-----------------+
|16842FL0070128-03|
|14002TN0400104-06|
|19722NM0010001-02|
|81413WI0460015-01|
|52697WI0010002-00|
+-----------------+
only showing top 5 rows



## Transformación


In [61]:
# T0 CREAR ATRIBUTOS SINTETICOS PARA EL EJERCICIO ACADÉMICO 
planes = planes.withColumn("NombrePlan",F.concat(F.lit("Plan_"), F.col("idPlan_T")))
planes = planes.withColumn("TipoPlan",F.when((F.hash("idPlan_T") % 3) == 0, "Tipo1")
                           .when((F.hash("idPlan_T") % 3) == 1, "Tipo2").otherwise("Tipo3"))
planes = planes.withColumn("NivelPlan",F.when((F.hash("idPlan_T") % 3) == 0, "Nivel1")
                           .when((F.hash("idPlan_T") % 3) == 1, "Nivel2").otherwise("Nivel3"))
planes = planes.withColumn("EstadoPlan",F.when((F.hash("idPlan_T") % 3) == 0, "Activo")
     .when((F.hash("idPlan_T") % 3) == 1, "Inactivo").otherwise("Suspendido"))

In [62]:
# T1. Renombrar IdPlan_T como IdPlanNatural
planes = planes.withColumn("IdPlanNatural", F.col("idPlan_T"))
# T2. Asignar surrogate key 
planes = planes.coalesce(1).withColumn("SKPlan",F.monotonically_increasing_id() + 1)

# T3. Crear columna FechaDesde con la fecha de hoy 
planes = planes.withColumn("FechaDesde", F.current_date())

#T4. Crear columna FechaHasta
planes = planes.withColumn("FechaHasta", F.lit(None).cast("date"))

#T5. Crear EsActual
# Como todos están vigentes (porque no tenemos los datos reales), se llena con 1
planes = planes.withColumn("EsActual", F.lit(1))

planes.show(5)



+-----------------+--------------------+--------+---------+----------+-----------------+------+----------+----------+--------+
|         idPlan_T|          NombrePlan|TipoPlan|NivelPlan|EstadoPlan|    IdPlanNatural|SKPlan|FechaDesde|FechaHasta|EsActual|
+-----------------+--------------------+--------+---------+----------+-----------------+------+----------+----------+--------+
|16842FL0070128-03|Plan_16842FL00701...|   Tipo2|   Nivel2|  Inactivo|16842FL0070128-03|     1|2025-11-20|      null|       1|
|14002TN0400104-06|Plan_14002TN04001...|   Tipo1|   Nivel1|    Activo|14002TN0400104-06|     2|2025-11-20|      null|       1|
|19722NM0010001-02|Plan_19722NM00100...|   Tipo2|   Nivel2|  Inactivo|19722NM0010001-02|     3|2025-11-20|      null|       1|
|81413WI0460015-01|Plan_81413WI04600...|   Tipo2|   Nivel2|  Inactivo|81413WI0460015-01|     4|2025-11-20|      null|       1|
|52697WI0010002-00|Plan_52697WI00100...|   Tipo1|   Nivel1|    Activo|52697WI0010002-00|     5|2025-11-20|     

In [63]:
# T6.Si cambia algún atributo del plan cerrar el registro anterior (actualizar FechaHasta y EsActual = 0) y crear uno nuevo
atributos_cambio = ["NombrePlan", "TipoPlan", "NivelPlan", "EstadoPlan"]
w = Window.partitionBy("idPlan_T").orderBy("FechaDesde")

# Detectar cambios y actualizar FechaHasta y EsActual
planes = planes.withColumn("FechaHasta_temp", F.lead("FechaDesde").over(w))
planes = planes.withColumn("FechaHasta_temp", F.expr("date_sub(FechaHasta_temp, 1)"))
planes = planes.withColumn("EsCambio", F.when(
F.lag(F.concat_ws("||", *atributos_cambio)).over(w) != F.concat_ws("||", *atributos_cambio),1).otherwise(0))

planes = planes.withColumn("FechaHasta",F.when(F.col("EsCambio") == 1,
                                               F.col("FechaHasta_temp")).otherwise(F.lit(None).cast("date")))

planes = planes.withColumn("EsActual",F.when(F.col("FechaHasta").isNull(), 1).otherwise(0))

# Eliminar columnas temporales
planes = planes.drop("FechaHasta_temp", "EsCambio")



In [64]:
planes = planes.drop("idPlan_T")
planes.show(5)

+--------------------+--------+---------+----------+-----------------+------+----------+----------+--------+
|          NombrePlan|TipoPlan|NivelPlan|EstadoPlan|    IdPlanNatural|SKPlan|FechaDesde|FechaHasta|EsActual|
+--------------------+--------+---------+----------+-----------------+------+----------+----------+--------+
|Plan_10207VA03800...|   Tipo3|   Nivel3|Suspendido|10207VA0380001-06|  2259|2025-11-20|      null|       1|
|Plan_10207VA03800...|   Tipo3|   Nivel3|Suspendido|10207VA0380001-06|  2740|2025-11-20|      null|       1|
|Plan_10207VA03800...|   Tipo3|   Nivel3|Suspendido|10207VA0380001-06|  5331|2025-11-20|      null|       1|
|Plan_10207VA03800...|   Tipo3|   Nivel3|Suspendido|10207VA0380001-06|  5910|2025-11-20|      null|       1|
|Plan_10207VA03800...|   Tipo3|   Nivel3|Suspendido|10207VA0380001-06|  8732|2025-11-20|      null|       1|
+--------------------+--------+---------+----------+-----------------+------+----------+----------+--------+
only showing top 5 

In [65]:
guardar_db(
    dest_db_connection_string,
    planes,
    'Proyecto_202515_G9.DimPlan',
    db_user,
    db_psswd
)

Py4JJavaError: An error occurred while calling o1607.save.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 84.0 failed 1 times, most recent failure: Lost task 0.0 in stage 84.0 (TID 480) (172.24.101.109 executor driver): java.sql.BatchUpdateException: Column 'FechaHasta' cannot be null
	at sun.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
	at sun.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
	at sun.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
	at java.lang.reflect.Constructor.newInstance(Constructor.java:423)
	at com.mysql.cj.util.Util.handleNewInstance(Util.java:192)
	at com.mysql.cj.util.Util.getInstance(Util.java:167)
	at com.mysql.cj.util.Util.getInstance(Util.java:174)
	at com.mysql.cj.jdbc.exceptions.SQLError.createBatchUpdateException(SQLError.java:224)
	at com.mysql.cj.jdbc.ClientPreparedStatement.executeBatchSerially(

# Dimensión Proveedor
## Extracción 
En esta dimensión se propuso usar campos que no están en la base de datos pero consideramos que deben existir y que son indispensables para los análisis solicitados por la compañía. En el ejercicio con la empresa se consultaría a la misma y se completaría la base de datos, sin embargo, al ser este un ejercicio académico y no tener contacto con la compañía para solicitar la información se decidió crear sinteticamente con el objetivo de desarrollar el proceso de ETL completo

In [66]:
sql_proveedor = '''
(
    SELECT idProveedor_T
    FROM FuentePlanesBeneficio_ETL
) AS Temp_proveedor
'''

proveedor = obtener_dataframe_de_bd(
    source_db_connection_string,
    sql_proveedor,
    db_user,
    db_psswd
)

proveedor.show(5)

+-------------+
|idProveedor_T|
+-------------+
|        16842|
|        14002|
|        19722|
|        81413|
|        52697|
+-------------+
only showing top 5 rows



In [67]:
# T0. Asignar datos sintéticos 
proveedor = proveedor.withColumn("NombreProveedor",F.concat(F.lit("Proveedor_"), F.col("idProveedor_T")))

proveedor = proveedor.withColumn("TipoProveedor",F.when((F.hash("idProveedor_T") % 3) == 0, "Tipo1")
     .when((F.hash("idProveedor_T") % 3) == 1, "Tipo2").otherwise("Tipo3"))

proveedor = proveedor.withColumn("CategoriaProveedor",F.when((F.hash("idProveedor_T") % 3) == 0, "CatA")
     .when((F.hash("idProveedor_T") % 3) == 1, "CatB").otherwise("CatC"))

proveedor = proveedor.withColumn("EstadoProveedor",F.when((F.hash("idProveedor_T") % 3) == 0, "Activo")
     .when((F.hash("idProveedor_T") % 3) == 1, "Inactivo").otherwise("Suspendido"))

 # T1.Renombrar idProveedor_T como idProveedorNatural
proveedor = proveedor.withColumnRenamed("idProveedor_T", "IdProveedorNatural")

# T2. Identificar y eliminar duplicados en las columnas NombreProveedor y IdProveedor_T
proveedor = proveedor.dropDuplicates(["IdProveedorNatural", "NombreProveedor"])
# T3. Asignar Surrogate Key bajo el nombre de SKProveedor
proveedor = proveedor.coalesce(1).withColumn("SKProveedor",F.monotonically_increasing_id() + 1)

proveedor.show(5)


+------------------+---------------+-------------+------------------+---------------+-----------+
|IdProveedorNatural|NombreProveedor|TipoProveedor|CategoriaProveedor|EstadoProveedor|SKProveedor|
+------------------+---------------+-------------+------------------+---------------+-----------+
|             10207|Proveedor_10207|        Tipo3|              CatC|     Suspendido|          1|
|             11269|Proveedor_11269|        Tipo1|              CatA|         Activo|          2|
|             11469|Proveedor_11469|        Tipo3|              CatC|     Suspendido|          3|
|             11512|Proveedor_11512|        Tipo1|              CatA|         Activo|          4|
|             12303|Proveedor_12303|        Tipo1|              CatA|         Activo|          5|
+------------------+---------------+-------------+------------------+---------------+-----------+
only showing top 5 rows



In [68]:
guardar_db(
    dest_db_connection_string,
    proveedor,
    'Proyecto_202515_G9.DimProveedor',
    db_user,
    db_psswd
)

# ETL tabla de hechos históricos
Se crearon 3 tablas de hechos asociadas a las dimensiones TipoBeneficio, Plan y Geografia. Para las 3 se siguió el mismo proceso inicialmente se cargaron las dimensiones con sus atributos historicos correspondientes, FechaDesde, FechaHasta, EsActual y sus correspondientes identificadores y llaves sustitutas posteriormente se identificaron los registros que habian sufrido algún cambio, se marcaron con 1 en una nueva etiqueta llamada cambio y se crearon los campos de FechaInicio y FechaFin para relacionar su vigencia a partir de FechaDesde y FechaHasta

In [69]:
# Cargar dimensiones necesarias
dim_benef = obtener_dataframe_de_bd(
    dest_db_connection_string,
    "Proyecto_202515_G9.DimTipoBeneficio",
    db_user,
    db_psswd
)

dim_plan = obtener_dataframe_de_bd(
    dest_db_connection_string,
    "Proyecto_202515_G9.DimPlan",
    db_user,
    db_psswd
)

dim_geo = obtener_dataframe_de_bd(
    dest_db_connection_string,
    "Proyecto_202515_G9.DimGeografia",
    db_user,
    db_psswd
)

dim_fecha = obtener_dataframe_de_bd(
    dest_db_connection_string,
    "Proyecto_202515_G9.DimFecha",
    db_user,
    db_psswd
)


# HISTÓRICOS TIPO BENEFICIO

df_hist_benef = dim_benef.filter("EsActual = 0").select(
    "IdTipoBeneficioNatural",
    "FechaDesde",
    "FechaHasta"
).withColumn("Cambio", F.lit(1))

df_hist_benef = df_hist_benef.join(
    dim_benef.select("SKTipoBeneficio", "IdTipoBeneficioNatural"),
    "IdTipoBeneficioNatural",
    "left"
)

df_hist_benef = df_hist_benef.join(
    dim_fecha.select("SKFecha", "Fecha"),
    dim_fecha.Fecha == df_hist_benef.FechaDesde,
    "left"
).withColumnRenamed("SKFecha", "SKFechaInicio").drop("Fecha")

df_hist_benef = df_hist_benef.join(
    dim_fecha.select("SKFecha", "Fecha"),
    dim_fecha.Fecha == df_hist_benef.FechaHasta,
    "left"
).withColumnRenamed("SKFecha", "SKFechaFin").drop("Fecha")


# HISTÓRICOS PLAN

df_hist_plan = dim_plan.filter("EsActual = 0").select(
    "IdPlanNatural",
    "FechaDesde",
    "FechaHasta"
).withColumn("Cambio", F.lit(1))

df_hist_plan = df_hist_plan.join(
    dim_plan.select("SKPlan", "IdPlanNatural"),
    "IdPlanNatural",
    "left"
)

df_hist_plan = df_hist_plan.join(
    dim_fecha.select("SKFecha", "Fecha"),
    dim_fecha.Fecha == df_hist_plan.FechaDesde,
    "left"
).withColumnRenamed("SKFecha", "SKFechaInicio").drop("Fecha")

df_hist_plan = df_hist_plan.join(
    dim_fecha.select("SKFecha", "Fecha"),
    dim_fecha.Fecha == df_hist_plan.FechaHasta,
    "left"
).withColumnRenamed("SKFecha", "SKFechaFin").drop("Fecha")


##########################################################
# HISTÓRICOS GEOGRAFIA
##########################################################

df_hist_geo = dim_geo.filter("EsActual = 0").select(
    "IdGeografiaNatural",
    "FechaDesde",
    "FechaHasta"
).withColumn("Cambio", F.lit(1))

df_hist_geo = df_hist_geo.join(
    dim_geo.select("SKGeografia", "IdGeografiaNatural"),
    "IdGeografiaNatural",
    "left"
)

df_hist_geo = df_hist_geo.join(
    dim_fecha.select("SKFecha", "Fecha"),
    dim_fecha.Fecha == df_hist_geo.FechaDesde,
    "left"
).withColumnRenamed("SKFecha", "SKFechaInicio").drop("Fecha")

df_hist_geo = df_hist_geo.join(
    dim_fecha.select("SKFecha", "Fecha"),
    dim_fecha.Fecha == df_hist_geo.FechaHasta,
    "left"
).withColumnRenamed("SKFecha", "SKFechaFin").drop("Fecha")




In [70]:
def guardar_db_lotes(db_connection_string, df, tabla, db_user, db_psswd, batch_size=5000):
    (
        df.write.format('jdbc')
        .mode('append')
        .option('url', db_connection_string)
        .option('dbtable', tabla)
        .option('user', db_user)
        .option('password', db_psswd)
        .option('driver', 'com.mysql.cj.jdbc.Driver')
        .option('batchsize', batch_size)  
        .option('rewriteBatchedStatements', 'true')  
        .save()
    )
    
guardar_db_lotes(
    dest_db_connection_string,
    df_hist_plan,
    "Proyecto_202515_G9.HechoHistPlan",
    db_user,
    db_psswd
)
print("Creada")

Creada


In [71]:
def guardar_db_lotes(db_connection_string, df, tabla, db_user, db_psswd, batch_size=5000):
    (
        df.write.format('jdbc')
        .mode('append')
        .option('url', db_connection_string)
        .option('dbtable', tabla)
        .option('user', db_user)
        .option('password', db_psswd)
        .option('driver', 'com.mysql.cj.jdbc.Driver')
        .option('batchsize', batch_size)  
        .option('rewriteBatchedStatements', 'true')  
        .save()
    )
    
guardar_db_lotes(
    dest_db_connection_string,
    df_hist_geo,
    "Proyecto_202515_G9.HechoHistGeografia",
    db_user,
    db_psswd
)
print("ok")

ok


In [72]:
def guardar_db_lotes(db_connection_string, df, tabla, db_user, db_psswd, batch_size=5000):
    (
        df.write.format('jdbc')
        .mode('append')
        .option('url', db_connection_string)
        .option('dbtable', tabla)
        .option('user', db_user)
        .option('password', db_psswd)
        .option('driver', 'com.mysql.cj.jdbc.Driver')
        .option('batchsize', batch_size)  
        .option('rewriteBatchedStatements', 'true')  
        .save()
    )
    
guardar_db_lotes(
    dest_db_connection_string,
    df_hist_benef,
    "Proyecto_202515_G9.HechoHistCondicionesTiposBeneficio",
    db_user,
    db_psswd
)
print("ok")

AnalysisException: Column "IdTipoBeneficioNatural" not found in schema Some(StructType(StructField(SKFecha,IntegerType,true), StructField(SKTipoBeneficio,LongType,true), StructField(SKMiniCondicion,LongType,true), StructField(Cambio,IntegerType,false)))

# Tabla hechos

In [73]:

# 1. CARGA DE FUENTE
tabla_planes = "FuentePlanesBeneficio_ETL"

df_planes = obtener_dataframe_de_bd(
    source_db_connection_string,
    tabla_planes,
    db_user,
    db_psswd
)

# 2. CARGA DE DIMENSIONES
dim_area   = obtener_dataframe_de_bd(dest_db_connection_string, "Proyecto_202515_G9.DimAreaServicio", db_user, db_psswd)
dim_plan   = obtener_dataframe_de_bd(dest_db_connection_string, "Proyecto_202515_G9.DimPlan", db_user, db_psswd)
dim_benef  = obtener_dataframe_de_bd(dest_db_connection_string, "Proyecto_202515_G9.DimTipoBeneficio", db_user, db_psswd)
dim_cond   = obtener_dataframe_de_bd(dest_db_connection_string, "Proyecto_202515_G9.DimCondicionPago", db_user, db_psswd)
dim_fecha  = obtener_dataframe_de_bd(dest_db_connection_string, "Proyecto_202515_G9.DimFecha", db_user, db_psswd)
dim_prov   = obtener_dataframe_de_bd(dest_db_connection_string, "Proyecto_202515_G9.DimProveedor", db_user, db_psswd)

# 3. NORMALIZACIÓN DE FECHA
df_planes = df_planes.withColumn(
    "FechaEmision",
    F.when(F.length("Fecha") == 4, F.concat(F.col("Fecha"), F.lit("-01-01")))
     .otherwise(F.col("Fecha"))
).withColumn("FechaEmision", F.to_date("FechaEmision"))

print(dim_area.count(), dim_area.dropDuplicates(["IdAreaServicioNatural"]).count())
print(dim_plan.count(), dim_plan.dropDuplicates(["IdPlanNatural"]).count())
print(dim_benef.count(), dim_benef.dropDuplicates(["IdTipoBeneficioNatural"]).count())
print(dim_cond.count(), dim_cond.dropDuplicates(["IdCondicionPagoNatural"]).count())
print(dim_prov.count(), dim_prov.dropDuplicates(["IdProveedorNatural"]).count())
print(dim_fecha.count(), dim_fecha.dropDuplicates(["Fecha"]).count())


11620 11620
654 654
898 205
261 21
514 172
1826 1826


En el anterior print, el primer número corresponde a la cantidad total de registros que hay en cada dimensión y el segundo a la cantidad de registros únicos. A través de este identificó gran cantidad de datos duplicados, como en las dimensiones Area, plan, beneficio, condicion tipo beneficio, proveedor se tienen datos datos descriptivos, los duplicados no aportan ningún tipo de información adicional al análisis por lo que se decidió eliminarlos para optimizar la tabla 

In [74]:
dim_area  = dim_area.dropDuplicates(["IdAreaServicioNatural"])
dim_plan  = dim_plan.dropDuplicates(["IdPlanNatural"])
dim_benef = dim_benef.dropDuplicates(["IdTipoBeneficioNatural"])
dim_cond  = dim_cond.dropDuplicates(["IdCondicionPagoNatural"])
dim_prov  = dim_prov.dropDuplicates(["IdProveedorNatural"])

In [75]:
# 4. FK FECHA
df_planes = df_planes.join(
    dim_fecha.select("SKFecha", "Fecha"),
    df_planes.FechaEmision == dim_fecha.Fecha,
    "left"
).drop(dim_fecha.Fecha)

# 5. FK AREA SERVICIO
df_planes = df_planes.join(
    dim_area.select("SKAreaServicio", "IdAreaServicioNatural"),
    df_planes.IdAreaDeServicio_T == dim_area.IdAreaServicioNatural,
    "left"
).drop("IdAreaDeServicio_T", "IdAreaServicioNatural")

df_planes = df_planes.withColumn(
    "SKAreaServicio",
    F.coalesce(F.col("SKAreaServicio"), F.lit(0))
)

# 6. FK TIPO BENEFICIO
df_planes = df_planes.join(
    dim_benef.select("SKTipoBeneficio", "IdTipoBeneficioNatural"),
    df_planes.IdTipoBeneficio_T == dim_benef.IdTipoBeneficioNatural,
    "left"
).drop("IdTipoBeneficioNatural")

# 7. FK CONDICIONES DE PAGO (UNA SOLA COLUMNA)
# Obtener MATCH por separado
df_planes = df_planes.join(
    dim_cond.select("SKCondicionPago", "IdCondicionPagoNatural"),
    df_planes.IdCondicionDePagoCopago_T == dim_cond.IdCondicionPagoNatural,
    "left"
).withColumnRenamed("SKCondicionPago", "SKCondicionPagoCopago") \
 .drop("IdCondicionPagoNatural")

df_planes = df_planes.join(
    dim_cond.select("SKCondicionPago", "IdCondicionPagoNatural"),
    df_planes.IdCondicionDePagoCoseguro_T == dim_cond.IdCondicionPagoNatural,
    "left"
).withColumnRenamed("SKCondicionPago", "SKCondicionPagoCoseguro") \
 .drop("IdCondicionPagoNatural")

# Crear COLUMNA ÚNICA final → la que SÍ existe en MySQL
df_planes = df_planes.withColumn(
    "SKCondicionPago",
    F.coalesce(F.col("SKCondicionPagoCopago"), F.col("SKCondicionPagoCoseguro"), F.lit(0))
)

df_planes = df_planes.drop(
    "IdCondicionDePagoCopago_T",
    "IdCondicionDePagoCoseguro_T",
    "SKCondicionPagoCopago",
    "SKCondicionPagoCoseguro"
)

# 8. FK PLAN
df_planes = df_planes.join(
    dim_plan.select("SKPlan", "IdPlanNatural"),
    df_planes.IdPlan_T == dim_plan.IdPlanNatural,
    "left"
).drop("IdPlanNatural")

# 9. FK PROVEEDOR
df_planes = df_planes.join(
    dim_prov.select("SKProveedor", "IdProveedorNatural"),
    df_planes.IdProveedor_T == dim_prov.IdProveedorNatural,
    "left"
).drop("IdProveedorNatural")

# 10. MEDIDAS
df_planes = df_planes.withColumn(
    "CantidadLimite",
    F.when((F.col("CantidadLimite").isNull()) | (F.col("CantidadLimite") == 0), 333)
     .otherwise(F.col("CantidadLimite"))
)

# 11. TABLA DE HECHO FINAL (COMPATIBLE CON MYSQL)
df_hecho = df_planes.select(
    "SKProveedor",
    "SKAreaServicio",
    "SKPlan",
    "SKTipoBeneficio",
    "SKCondicionPago",     # <--- SOLO ESTA
    "SKFecha",
    F.col("valorCopago").cast("double").alias("valorCopago"),
    F.col("valorCoseguro").cast("double").alias("valorCoseguro"),
    F.col("CantidadLimite").cast("double").alias("CantidadLimite")
)

# 12. VALIDACIÓN DEL RESULTADO

print ("\nVALIDACION DEL RESULTADO")
print("\n1. SCHEMA FINAL")
df_hecho.printSchema()

print("\n2. PRIMERAS FILAS")
df_hecho.show(20, truncate=False)


VALIDACION DEL RESULTADO

1. SCHEMA FINAL
root
 |-- SKProveedor: long (nullable = true)
 |-- SKAreaServicio: long (nullable = false)
 |-- SKPlan: long (nullable = true)
 |-- SKTipoBeneficio: long (nullable = true)
 |-- SKCondicionPago: integer (nullable = false)
 |-- SKFecha: integer (nullable = true)
 |-- valorCopago: double (nullable = true)
 |-- valorCoseguro: double (nullable = true)
 |-- CantidadLimite: double (nullable = true)


2. PRIMERAS FILAS
+-------------+--------------+------+---------------+---------------+--------+-----------+-------------+--------------+
|SKProveedor  |SKAreaServicio|SKPlan|SKTipoBeneficio|SKCondicionPago|SKFecha |valorCopago|valorCoseguro|CantidadLimite|
+-------------+--------------+------+---------------+---------------+--------+-----------+-------------+--------------+
|352187318272 |8589935056    |217   |210            |14             |20201231|0.0        |100.0        |333.0         |
|34359738368  |17179870201   |260   |243            |12       

In [76]:
from pyspark.sql.types import *

def guardar_db_lotes(db_connection_string, df, tabla, db_user, db_psswd, batch_size=5000):
    (
        df.write.format('jdbc')
        .mode('append')
        .option('url', db_connection_string)
        .option('dbtable', tabla)
        .option('user', db_user)
        .option('password', db_psswd)
        .option('driver', 'com.mysql.cj.jdbc.Driver')
        .option('batchsize', batch_size)  
        .option('rewriteBatchedStatements', 'true')  
        .save()
    )

print("Iniciando carga por lotes...")


guardar_db_lotes(
    dest_db_connection_string,
    df_hecho,
    "Proyecto_202515_G9.HechoPlanesTiposBeneficio",
    db_user,
    db_psswd,
    batch_size=1000
)

print("Proceso completado.")



Iniciando carga por lotes...
Proceso completado.
